# Práctica 1 – Clasificación de personajes de Los Simpson con redes convolucionales

Aprendizaje Automático II · 2026-27

## 1. Librerías y configuración

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from tqdm.auto import tqdm

In [ ]:
SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
torch.cuda.manual_seed_all(SEMILLA)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODO_PRUEBA = device.type == "cpu"

print("Dispositivo:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
print("Modo prueba:", MODO_PRUEBA)

In [ ]:
DIR_DATOS = Path("data")
DIR_TRAIN = DIR_DATOS / "simpsons_dataset"
DIR_TEST = DIR_DATOS / "kaggle_simpson_testset" / "kaggle_simpson_testset"
DIR_RESULTADOS = Path("resultados")
DIR_RESULTADOS.mkdir(exist_ok=True)

PERSONAJES = [
    "homer_simpson",
    "ned_flanders",
    "moe_szyslak",
    "lisa_simpson",
    "bart_simpson",
    "marge_simpson",
    "krusty_the_clown",
    "principal_skinner",
    "charles_montgomery_burns",
    "milhouse_van_houten",
]
NUM_CLASES = len(PERSONAJES)
CLASE_A_IDX = {clase: i for i, clase in enumerate(PERSONAJES)}
NOMBRES = [c.replace("_", " ").title() for c in PERSONAJES]

TAM_IMG = 64 if MODO_PRUEBA else 128
MAX_POR_CLASE = 100 if MODO_PRUEBA else None
PROP_VALIDACION = 0.15

## 2. Rutas de las imágenes y división en entrenamiento, validación y test

In [ ]:
def rutas_entrenamiento():
    rutas, etiquetas = [], []
    for clase in PERSONAJES:
        archivos = sorted((DIR_TRAIN / clase).glob("*.jpg"))
        if MAX_POR_CLASE:
            archivos = random.Random(SEMILLA).sample(archivos, min(MAX_POR_CLASE, len(archivos)))
        rutas += archivos
        etiquetas += [CLASE_A_IDX[clase]] * len(archivos)
    return rutas, etiquetas


def rutas_test():
    rutas, etiquetas = [], []
    for archivo in sorted(DIR_TEST.glob("*.jpg")):
        clase = archivo.stem.rsplit("_", 1)[0]
        if clase in CLASE_A_IDX:
            rutas.append(archivo)
            etiquetas.append(CLASE_A_IDX[clase])
    return rutas, etiquetas


rutas, etiquetas = rutas_entrenamiento()
rutas_tr, rutas_val, y_tr, y_val = train_test_split(
    rutas, etiquetas, test_size=PROP_VALIDACION, stratify=etiquetas, random_state=SEMILLA
)
rutas_te, y_te = rutas_test()

print(f"Entrenamiento: {len(rutas_tr)}  |  Validación: {len(rutas_val)}  |  Test: {len(rutas_te)}")

In [ ]:
conteo = pd.DataFrame({
    "Entrenamiento": pd.Series(y_tr).value_counts(),
    "Validación": pd.Series(y_val).value_counts(),
    "Test": pd.Series(y_te).value_counts(),
}).sort_index()
conteo.index = NOMBRES
conteo.loc["TOTAL"] = conteo.sum()
conteo

In [ ]:
ax = conteo.drop("TOTAL").plot(kind="barh", stacked=True, figsize=(9, 5))
ax.invert_yaxis()
ax.set_xlabel("Número de imágenes")
ax.set_title("Imágenes por personaje y conjunto")
plt.tight_layout()
plt.savefig(DIR_RESULTADOS / "distribucion_clases.png", dpi=150)
plt.show()

In [ ]:
fig, ejes = plt.subplots(2, 5, figsize=(15, 6))
for eje, clase in zip(ejes.flat, PERSONAJES):
    ruta = next(r for r, y in zip(rutas_tr, y_tr) if y == CLASE_A_IDX[clase])
    imagen = Image.open(ruta)
    eje.imshow(imagen)
    eje.set_title(f"{clase.replace('_', ' ').title()}\n{imagen.size[0]}x{imagen.size[1]}", fontsize=10)
    eje.axis("off")
plt.tight_layout()
plt.savefig(DIR_RESULTADOS / "ejemplos_personajes.png", dpi=150)
plt.show()

## 3. Carga de las imágenes en memoria y normalización

In [ ]:
def cargar_imagenes(rutas, tam):
    imagenes = torch.empty((len(rutas), 3, tam, tam), dtype=torch.uint8)
    for i, ruta in enumerate(tqdm(rutas, desc="Cargando imágenes")):
        imagen = Image.open(ruta).convert("RGB").resize((tam, tam), Image.Resampling.BILINEAR)
        imagenes[i] = torch.from_numpy(np.array(imagen)).permute(2, 0, 1)
    return imagenes


def cargar_con_cache(nombre, rutas, etiquetas):
    modo = "prueba" if MODO_PRUEBA else "completo"
    archivo = DIR_DATOS / f"cache_{nombre}_{TAM_IMG}px_{modo}.pt"
    if archivo.exists():
        return torch.load(archivo)
    X = cargar_imagenes(rutas, TAM_IMG)
    y = torch.tensor(etiquetas, dtype=torch.long)
    torch.save((X, y), archivo)
    return X, y


X_tr, y_tr_t = cargar_con_cache("train", rutas_tr, y_tr)
X_val, y_val_t = cargar_con_cache("val", rutas_val, y_val)
X_te, y_te_t = cargar_con_cache("test", rutas_te, y_te)

print("Entrenamiento:", tuple(X_tr.shape), "| Validación:", tuple(X_val.shape), "| Test:", tuple(X_te.shape))
print(f"Memoria ocupada: {(X_tr.nbytes + X_val.nbytes + X_te.nbytes) / 1e9:.2f} GB")

In [ ]:
suma = torch.zeros(3)
suma_cuadrados = torch.zeros(3)
n_pixeles = 0
for lote in X_tr.split(1000):
    x = lote.float() / 255
    suma += x.sum(dim=(0, 2, 3))
    suma_cuadrados += (x ** 2).sum(dim=(0, 2, 3))
    n_pixeles += x.shape[0] * x.shape[2] * x.shape[3]

MEDIA = (suma / n_pixeles).tolist()
DESVIACION = (suma_cuadrados / n_pixeles - (suma / n_pixeles) ** 2).sqrt().tolist()

print("Media por canal (R, G, B):     ", [round(m, 4) for m in MEDIA])
print("Desviación por canal (R, G, B):", [round(d, 4) for d in DESVIACION])

## 4. Dataset y DataLoaders

In [ ]:
class DatasetSimpsons(Dataset):
    def __init__(self, imagenes, etiquetas, transformacion):
        self.imagenes = imagenes
        self.etiquetas = etiquetas
        self.transformacion = transformacion

    def __len__(self):
        return len(self.etiquetas)

    def __getitem__(self, i):
        return self.transformacion(self.imagenes[i]), self.etiquetas[i]


transformacion_base = v2.Compose([
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEDIA, DESVIACION),
])

ds_tr = DatasetSimpsons(X_tr, y_tr_t, transformacion_base)
ds_val = DatasetSimpsons(X_val, y_val_t, transformacion_base)
ds_te = DatasetSimpsons(X_te, y_te_t, transformacion_base)


def crear_loaders(tam_lote):
    loader_tr = DataLoader(ds_tr, batch_size=tam_lote, shuffle=True)
    loader_val = DataLoader(ds_val, batch_size=tam_lote, shuffle=False)
    loader_te = DataLoader(ds_te, batch_size=tam_lote, shuffle=False)
    return loader_tr, loader_val, loader_te

In [ ]:
loader_tr, loader_val, loader_te = crear_loaders(32)
x, y = next(iter(loader_tr))
print("Forma del lote de imágenes:", tuple(x.shape), x.dtype)
print("Forma del lote de etiquetas:", tuple(y.shape), y.dtype)
print(f"Media del lote: {x.mean():.3f}  |  Desviación del lote: {x.std():.3f}")

media_t = torch.tensor(MEDIA).view(3, 1, 1)
desv_t = torch.tensor(DESVIACION).view(3, 1, 1)
fig, ejes = plt.subplots(2, 8, figsize=(16, 4.5))
for eje, imagen, etiqueta in zip(ejes.flat, x, y):
    eje.imshow((imagen * desv_t + media_t).clamp(0, 1).permute(1, 2, 0))
    eje.set_title(NOMBRES[etiqueta], fontsize=8)
    eje.axis("off")
plt.suptitle(f"Lote de entrenamiento ya preprocesado ({TAM_IMG}x{TAM_IMG})")
plt.tight_layout()
plt.show()